# Dynamic Predictions (Python)

**Dmitris Rizopoulos**

This notebook demonstrates dynamic individualized predictions for longitudinal count data using the Python **glmmadaptive** package.
It is a companion to the R vignette `Dynamic_Predictions.Rmd`.

## Contents
1. Introduction
2. Data Simulation
3. Train / Test Split
4. Model Fitting (Poisson and ZI Negative Binomial)
5. Dynamic Predictions with Confidence Intervals
6. Visualising Dynamic Predictions
7. Proper Scoring Rules


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.special import expit

from glmmadaptive import MixedModel, scoring_rules
from glmmadaptive.families import Poisson, ZINegativeBinomial

print('glmmadaptive imports OK')

## 1. Introduction

In the setting of longitudinal data, **dynamic individualized predictions** are predictions for the longitudinal outcomes that are updated as extra measurements are recorded for a subject.

For a new subject $j$ with measurements $\mathcal{Y}_j(t)$ up to time $t$, we are interested in

$$\omega_j(u \mid t) = E\{y_j(u) \mid \mathcal{Y}_j(t), \mathcal{D}_n\}, \quad u > t.$$

We approximate this using Monte Carlo sampling from the Laplace approximation to the random-effects posterior $p(b_j \mid \mathcal{Y}_j(t), \hat\theta)$.


## 2. Data Simulation

We simulate longitudinal count data from a zero-inflated negative binomial (ZINB) model with random intercepts in both the count and zero-inflation parts.


In [ ]:
np.random.seed(12345)

n = 150        # subjects (R vignette uses 500; reduced here for speed)
K = 8          # measurements per subject
t_max = 4.0

DF = pd.DataFrame({
    'id':   np.repeat(np.arange(1, n + 1), K),
    'time': np.concatenate([
        np.concatenate([[0.0], np.sort(np.random.uniform(0, t_max, K - 1))])
        for _ in range(n)
    ]),
    'sex':  np.repeat(
        np.where(np.arange(1, n + 1) <= n // 2, 'male', 'female'), K
    ),
})
DF['sex'] = pd.Categorical(DF['sex'], categories=['male', 'female'])

# True parameters
betas  = np.array([0.8, -0.5, 0.8, -0.5])   # count part fixed effects
shape  = 2.0                                  # NB dispersion (size)
gammas = np.array([-2.5, 0.5])               # ZI part fixed effects
D11, D22 = 1.0, 0.8                          # RE variances

b_count = np.random.normal(0, np.sqrt(D11), n)
b_zi    = np.random.normal(0, np.sqrt(D22), n)

import patsy
X    = np.asarray(patsy.dmatrix('~ sex * time', DF, return_type='matrix'))
X_zi = np.asarray(patsy.dmatrix('~ sex', DF, return_type='matrix'))
Z    = np.ones((n * K, 1))
Z_zi = np.ones((n * K, 1))

id0    = DF['id'].values - 1
eta_y  = X @ betas  + (Z  * b_count[id0]).sum(1)
eta_zi = X_zi @ gammas + (Z_zi * b_zi[id0]).sum(1)

from scipy.stats import nbinom
mu_y = np.exp(eta_y)
y = nbinom.rvs(n=shape, p=shape / (shape + mu_y))
extra_zero = np.random.binomial(1, expit(eta_zi)).astype(bool)
y[extra_zero] = 0

DF['y'] = y.astype(int)

print(f'Dataset: {len(DF)} observations, {n} subjects')
print(f'Proportion of zeros: {(DF["y"] == 0).mean():.3f}')
DF.head(8)

## 3. Train / Test Split

We split into a training set (100 subjects) and a test set (50 subjects).


In [ ]:
rng_split = np.random.default_rng(42)
ids_train = np.sort(rng_split.choice(np.arange(1, n + 1), size=100, replace=False))

DF_train = DF[DF['id'].isin(ids_train)].copy()
DF_test  = DF[~DF['id'].isin(ids_train)].copy()

print(f'Training: {DF_train["id"].nunique()} subjects, {len(DF_train)} obs')
print(f'Test:     {DF_test["id"].nunique()} subjects, {len(DF_test)} obs')

## 4. Model Fitting

We fit two models:
- **`res1`**: Poisson GLMM — fixed effects `sex`, `time`, `sex:time` and random intercept.
- **`res2`**: ZI Negative Binomial GLMM — same count-part structure, plus `sex` fixed effect and random intercept for the zero-inflation part.


In [ ]:
# Poisson model
fm1 = MixedModel(
    fixed='y ~ sex * time',
    random='~ 1 | id',
    data=DF_train,
    family=Poisson(),
)
res1 = fm1.fit(verbose=False)
print(res1.summary())

In [ ]:
# Zero-inflated negative binomial model
fm2 = MixedModel(
    fixed='y ~ sex * time',
    random='~ 1 | id',
    data=DF_train,
    family=ZINegativeBinomial(),
    zi_fixed='~ sex',
    zi_random='~ 1 | id',
)
res2 = fm2.fit(verbose=False)
print(res2.summary())

In [ ]:
print(f'Poisson log-lik:  {res1.logLik:.2f}   AIC: {res1.aic:.2f}')
print(f'ZINB    log-lik:  {res2.logLik:.2f}   AIC: {res2.aic:.2f}')

## 5. Dynamic Predictions

We use measurements **before time 2** as the information period and predict for **time ≥ 2**.

`predict_dynamic()` takes:
- `newdata` — information period (used to estimate each subject's random effects via the Laplace posterior mode)
- `newdata2` — prediction period
- `se_fit=True` — Monte Carlo confidence intervals


In [ ]:
DF_test_info = DF_test[DF_test['time'] < 2].copy()
DF_test_pred = DF_test[DF_test['time'] >= 2].copy()

# Only keep subjects that appear in both periods
ids_both = set(DF_test_info['id'].unique()) & set(DF_test_pred['id'].unique())
DF_test_info = DF_test_info[DF_test_info['id'].isin(ids_both)].copy()
DF_test_pred = DF_test_pred[DF_test_pred['id'].isin(ids_both)].copy()

print(f'Subjects with data in both periods: {len(ids_both)}')

In [ ]:
# Dynamic predictions — Poisson
preds_fm1 = res1.predict_dynamic(
    newdata=DF_test_info,
    newdata2=DF_test_pred,
    se_fit=True, n_mc=200, seed=1,
)

# Dynamic predictions — ZINB
preds_fm2 = res2.predict_dynamic(
    newdata=DF_test_info,
    newdata2=DF_test_pred,
    se_fit=True, n_mc=200, seed=1,
)

# Combine both periods for ZINB
pred_data_fm2 = pd.concat(
    [preds_fm2['newdata'], preds_fm2['newdata2']], ignore_index=True
).sort_values(['id', 'time'])

print('Columns:', pred_data_fm2.columns.tolist())
pred_data_fm2[['id','time','y','pred','low','upp','zi_probs']].head(10)

## 6. Visualising Dynamic Predictions

The dashed vertical line at $t=2$ separates the information period (left) from the prediction period (right).


In [ ]:
np.random.seed(7)
plot_ids = np.random.choice(list(ids_both), size=min(9, len(ids_both)), replace=False)

fig, axes = plt.subplots(3, 3, figsize=(13, 9), sharey=False)
axes = axes.ravel()

for ax, sid in zip(axes, plot_ids):
    subj = pred_data_fm2[pred_data_fm2['id'] == sid]
    ax.scatter(subj['time'], subj['y'], color='black', s=18, zorder=5)
    ax.plot(subj['time'], subj['pred'], color='tab:red', lw=1.5)
    ax.fill_between(subj['time'], subj['low'], subj['upp'],
                    color='tab:red', alpha=0.2)
    ax.axvline(x=2.0, color='grey', linestyle='--', lw=1.0)
    ax.set_title(f'Subject {sid}', fontsize=9)
    ax.set_xlabel('Time', fontsize=8)
    ax.set_ylabel('Count', fontsize=8)
    ax.tick_params(labelsize=7)

handles = [
    plt.scatter([], [], color='black', s=15, label='Observed'),
    plt.Line2D([0], [0], color='tab:red', lw=1.5, label='Expected'),
    mpatches.Patch(color='tab:red', alpha=0.3, label='95% CI'),
    plt.Line2D([0], [0], color='grey', lw=1, ls='--', label='t=2 boundary'),
]
fig.legend(handles=handles, loc='lower center', ncol=4, fontsize=8, frameon=False,
           bbox_to_anchor=(0.5, -0.02))
plt.suptitle('Dynamic Predictions — ZI Negative Binomial Model', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

## 7. Proper Scoring Rules

We evaluate predictive accuracy using three proper scoring rules:

| Rule | Formula | Better |
|---|---|---|
| **Logarithmic** | $\log P(Y = y)$ | closer to 0 |
| **Quadratic** | $2P(Y=y) - \sum_k P(Y=k)^2$ | higher |
| **Spherical** | $P(Y=y)/\sqrt{\sum_k P(Y=k)^2}$ | closer to 1 |

The predictive PMF is computed by Monte Carlo averaging over the random-effects posterior.


In [ ]:
# Scoring rules for Poisson
scr_fm1 = scoring_rules(
    res1, newdata=DF_test_info, newdata2=DF_test_pred,
    max_count=300, n_mc=100, seed=1,
)

# Scoring rules for ZINB
scr_fm2 = scoring_rules(
    res2, newdata=DF_test_info, newdata2=DF_test_pred,
    max_count=300, n_mc=100, seed=1,
)

print('Scoring columns:', [c for c in scr_fm1.columns if c not in DF_test_pred.columns])
scr_fm1[['id','time','y','logarithmic','quadratic','spherical']].head()

In [ ]:
# Combine results
scoring_data = scr_fm1[['id','time','y','logarithmic','quadratic','spherical']].copy()
scoring_data = scoring_data.rename(columns={
    'logarithmic': 'log_pois', 'quadratic': 'quad_pois', 'spherical': 'sph_pois',
})
scoring_data['log_zinb']  = scr_fm2['logarithmic'].values
scoring_data['quad_zinb'] = scr_fm2['quadratic'].values
scoring_data['sph_zinb']  = scr_fm2['spherical'].values

print('Mean scoring rules across prediction-period observations')
print(f'{"Model":<20} {"Logarithmic":>14} {"Quadratic":>12} {"Spherical":>12}')
print('-' * 62)
for label, cols in [
    ('Poisson', ('log_pois', 'quad_pois', 'sph_pois')),
    ('ZI NegBin',  ('log_zinb', 'quad_zinb', 'sph_zinb')),
]:
    row = scoring_data[list(cols)].mean()
    print(f'{label:<20} {row.iloc[0]:>14.4f} {row.iloc[1]:>12.4f} {row.iloc[2]:>12.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

configs = [
    ('sph_pois', 'sph_zinb', 'Spherical scoring rule',
     'Spherical rule (higher = better)'),
    ('log_pois', 'log_zinb', 'Logarithmic scoring rule',
     'Logarithmic rule (closer to 0 = better)'),
]

for ax, (col1, col2, ylabel, title) in zip(axes, configs):
    ax.scatter(scoring_data['time'], scoring_data[col1],
               color='black', s=5, alpha=0.4, label='Poisson')
    ax.scatter(scoring_data['time'], scoring_data[col2],
               color='tab:red', s=5, alpha=0.4, label='ZI NegBin')
    for col, color in [(col1, 'black'), (col2, 'tab:red')]:
        srt = scoring_data.sort_values('time')
        ax.plot(
            srt['time'].rolling(30, min_periods=1, center=True).mean(),
            srt[col].rolling(30, min_periods=1, center=True).mean(),
            color=color, lw=2.0,
        )
    ax.set_xlabel('Follow-up time', fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=10)
    ax.legend(fontsize=9, markerscale=2)
    ax.axvline(2.0, color='grey', linestyle=':', lw=1)

plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:

1. Simulating zero-inflated negative binomial longitudinal count data.
2. Fitting a Poisson GLMM and a ZI negative binomial GLMM.
3. Computing dynamic predictions with `predict_dynamic()`, which:
   - Estimates subject random effects from historical data.
   - Provides Monte Carlo confidence intervals.
   - Returns structural-zero probabilities for ZI models.
4. Evaluating predictions with `scoring_rules()` (logarithmic, quadratic, spherical).

The ZINB model is expected to outperform Poisson when data exhibit excess zeros and overdispersion.
